# 03 · Training and evaluating the model

**Supports agenda block 7** ("Training & Evaluating the Model"). LightGBM
with walk-forward cross-validation - never a random shuffle-split, which
would train on the future and validate on the past - and the
**Information Coefficient (IC)**, HAC-corrected for the autocorrelation
that a 21-day-overlapping label mechanically introduces, as the metric
that actually says whether the model found anything.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

if "COLAB_RELEASE_TAG" in os.environ:
    repo_dir = Path(os.environ.get("PACKT_WORKSHOP_DIR", "/content/packt-workshop"))
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/ml4t/packt-workshop.git",
                repo_dir,
            ],
            check=True,
        )
    ready = repo_dir.parent / ".packt-workshop-ready"
    if not ready.exists():
        # Colab's kernel already has numpy imported before this cell runs. Installing a
        # different numpy from requirements-colab.txt overwrites the files on disk but not
        # the compiled submodules already cached in this process, so importing a numpy
        # submodule for the first time later (e.g. via ml4t.engineer) mixes the new files
        # with the stale cached extension and raises ImportError. Put numpy back to the
        # version this process already has loaded once the rest of the install is done.
        import numpy as _colab_numpy

        colab_numpy_version = _colab_numpy.__version__
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "-r",
                repo_dir / "requirements-colab.txt",
            ],
            check=True,
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--force-reinstall",
                "--no-deps",
                f"numpy=={colab_numpy_version}",
            ],
            check=True,
        )
        ready.touch()
    os.chdir(repo_dir / "notebooks")
    if not (repo_dir / "data/model_dataset.parquet").exists():
        print("Preparing the feature-and-label dataset required by this notebook.")
        subprocess.run([sys.executable, "02_features_labels.py"], check=True)

In [2]:
import lightgbm as lgb
import pandas as pd
from ml4t.diagnostic.metrics import compute_ic_hac_stats, cross_sectional_ic_series
from ml4t.diagnostic.splitters import WalkForwardConfig, WalkForwardCV

DATA_DIR = "../data"
LABEL_HORIZON = 21  # trading days - must match the label built in 02_features_labels
N_VALIDATION_FOLDS = 8
TRAIN_SIZE = "10Y"
VALIDATION_SIZE = "1Y"
HOLDOUT_START = "2024-01-01"
HOLDOUT_END = "2025-12-31"

dataset = pd.read_parquet(f"{DATA_DIR}/model_dataset.parquet")
feature_cols = [
    "mom_21d",
    "mom_63d",
    "mom_126d",
    "mom_252d",
    "vol_21d",
    "vol_63d",
    "rsi_14",
    "dollar_vol_rank",
]

dataset["timestamp"] = pd.to_datetime(dataset["timestamp"]).dt.tz_localize("UTC")
dataset = dataset.sort_values("timestamp").set_index("timestamp")
X, y = dataset[feature_cols], dataset["fwd_ret_21d"]

## Walk-forward cross-validation

The split matches the book's ETF case study: eight one-year validation
folds, a rolling 10-year training boundary, a 21-trading-day purge, and a
sealed holdout from 2024-01-01 through 2025-12-31. `label_horizon=21` tells
the splitter that a training row's label is only fully known 21 trading
days after its feature date. It therefore removes training rows whose
label window would otherwise overlap validation.

The splitter operates on unique trading dates. Passing the repeated panel
index directly would count one row per ETF rather than one session and
produce incorrect fold boundaries.

In [3]:
split_dates = pd.DatetimeIndex(X.index.unique()).sort_values()
split_frame = pd.DataFrame(index=split_dates)

cv_config = WalkForwardConfig(
    n_splits=N_VALIDATION_FOLDS,
    train_size=TRAIN_SIZE,
    test_size=VALIDATION_SIZE,
    label_horizon=LABEL_HORIZON,
    test_start=HOLDOUT_START,
    test_end=HOLDOUT_END,
    fold_direction="backward",
    calendar_id="NYSE",
)
cv = WalkForwardCV(config=cv_config)
cv.expanding = False

backward_splits = list(cv.split(split_frame))
validation_splits = list(reversed(backward_splits))
holdout_dates = split_dates[cv.test_indices_]

assert len(validation_splits) == N_VALIDATION_FOLDS
assert all(len(validation_idx) == 252 for _, validation_idx in validation_splits)
assert max(len(train_idx) for train_idx, _ in validation_splits) <= 10 * 252
assert all(
    split_dates[train_idx[-1]] < split_dates[validation_idx[0]]
    for train_idx, validation_idx in validation_splits
)
assert max(split_dates[validation_idx[-1]] for _, validation_idx in validation_splits) < min(
    holdout_dates
)

fold_predictions = []
fold_windows = []
for fold, (train_date_idx, validation_date_idx) in enumerate(validation_splits):
    train_dates = split_dates[train_date_idx]
    validation_dates = split_dates[validation_date_idx]
    train_rows = X.index.isin(train_dates)
    validation_rows = X.index.isin(validation_dates)

    X_train, y_train = X.loc[train_rows], y.loc[train_rows]
    X_validation = X.loc[validation_rows]

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=200,
        verbosity=-1,
    )
    model.fit(X_train, y_train)

    preds = dataset.loc[validation_rows, ["symbol", "fwd_ret_21d"]].copy()
    preds["prediction"] = model.predict(X_validation)
    preds["fold"] = fold
    fold_predictions.append(preds)

    fold_windows.append(
        {
            "fold": fold,
            "train_start": X_train.index.min().date(),
            "train_end": X_train.index.max().date(),
            "validation_start": X_validation.index.min().date(),
            "validation_end": X_validation.index.max().date(),
        }
    )

    print(
        f"fold {fold}: train {X_train.index.min().date()}..{X_train.index.max().date()} "
        f"({len(X_train):,} rows) -> validation "
        f"{X_validation.index.min().date()}..{X_validation.index.max().date()} "
        f"({len(X_validation):,} rows)"
    )

print(
    f"sealed holdout: {holdout_dates.min().date()}..{holdout_dates.max().date()} "
    f"({len(holdout_dates):,} trading days)"
)

pd.DataFrame(fold_windows)

fold 0: train 2008-01-03..2015-11-24 (134,540 rows) -> validation 2015-12-28..2016-12-23 (21,920 rows)


fold 1: train 2008-01-03..2016-11-23 (156,439 rows) -> validation 2016-12-27..2017-12-26 (22,420 rows)


fold 2: train 2008-01-03..2017-11-24 (178,817 rows) -> validation 2017-12-27..2018-12-27 (23,898 rows)


fold 3: train 2008-12-29..2018-11-26 (192,150 rows) -> validation 2018-12-28..2019-12-27 (24,187 rows)


fold 4: train 2009-12-29..2019-11-26 (202,964 rows) -> validation 2019-12-30..2020-12-28 (23,692 rows)


fold 5: train 2010-12-29..2020-11-25 (210,603 rows) -> validation 2020-12-29..2021-12-28 (23,920 rows)


fold 6: train 2011-12-28..2021-11-26 (217,147 rows) -> validation 2021-12-29..2022-12-28 (23,937 rows)


fold 7: train 2012-12-31..2022-11-28 (222,184 rows) -> validation 2022-12-29..2023-12-29 (24,148 rows)
sealed holdout: 2024-01-02..2025-12-01 (481 trading days)


,fold,train_start,train_end,validation_start,validation_end
0,0,2008-01-03,2015-11-24,2015-12-28,2016-12-23
1,1,2008-01-03,2016-11-23,2016-12-27,2017-12-26
2,2,2008-01-03,2017-11-24,2017-12-27,2018-12-27
3,3,2008-12-29,2018-11-26,2018-12-28,2019-12-27
4,4,2009-12-29,2019-11-26,2019-12-30,2020-12-28
5,5,2010-12-29,2020-11-25,2020-12-29,2021-12-28
6,6,2011-12-28,2021-11-26,2021-12-29,2022-12-28
7,7,2012-12-31,2022-11-28,2022-12-29,2023-12-29


## Visualizing the fold structure

The table above states the eight windows; this renders them. Each row is one
fold's training span (navy) advancing across the panel, its one-year
validation span (gold), and the sealed 2024-2025 holdout (orange) that no
fold ever touches.

In [4]:
from pathlib import Path

from ml4t.diagnostic.visualization.cv_plots import plot_cv_folds

FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(exist_ok=True)


def export_static(fig, filename, **size):
    """Save a Plotly figure as PNG.

    ml4t-diagnostic<=0.1.0b25's static-export path (kaleido>=1.0's orjson
    serializer) rejects any pd.Timestamp left in a bar trace's `base`, and
    plot_cv_folds does not yet declare its x-axis as a date axis. Both are
    fixed upstream (diagnostic commit 390207c) but not yet released; drop
    this once the pin moves past b25.
    """
    fig.update_xaxes(type="date")
    for trace in fig.data:
        base = getattr(trace, "base", None)
        if base is not None:
            values = base if isinstance(base, (list, tuple)) else [base]
            trace.base = tuple(v.isoformat() if isinstance(v, pd.Timestamp) else v for v in values)
    fig.write_image(FIGURE_DIR / filename, **size)
    return fig


cv_fig = plot_cv_folds(
    cv,
    X=split_frame,
    title="Walk-forward CV: 8 folds, rolling 10-year train, sealed holdout",
    theme="print",
)
export_static(cv_fig, "model-cv-folds.png", width=1400, height=650, scale=2)
cv_fig

## The Information Coefficient

Every prediction above came from a fold where the model never saw that
period during training - this is out-of-sample by construction, not by
promise. We pool the eight validation folds and compute the cross-sectional
Spearman IC per date, then HAC-correct the resulting t-statistic. The two
holdout years remain sealed and do not influence this model assessment.

In [5]:
oos = pd.concat(fold_predictions).reset_index().rename(columns={"index": "timestamp"})

ic_series = cross_sectional_ic_series(
    oos,
    oos,
    pred_col="prediction",
    ret_col="fwd_ret_21d",
    date_col="timestamp",
    entity_col="symbol",
    method="spearman",
)
print(ic_series.describe())

             n_obs           ic
count  2016.000000  2016.000000
mean     93.314484     0.015207
std       3.195889     0.226101
min      86.000000    -0.752792
25%      89.000000    -0.134452
50%      95.000000     0.013348
75%      95.000000     0.173398
max      96.000000     0.686878


In [6]:
hac_stats = compute_ic_hac_stats(ic_series, ic_col="ic", label_horizon=LABEL_HORIZON)
print(hac_stats)

assert len(fold_predictions) == N_VALIDATION_FOLDS
assert hac_stats["n_periods"] == N_VALIDATION_FOLDS * 252
assert abs(hac_stats["mean_ic"] - 0.0152068) < 1e-6
assert abs(hac_stats["naive_t_stat"] - 3.0198) < 1e-4
assert abs(hac_stats["t_stat"] - 0.9418) < 1e-4
assert holdout_dates.min() >= pd.Timestamp(HOLDOUT_START, tz="UTC")
assert holdout_dates.max() <= pd.Timestamp(HOLDOUT_END, tz="UTC")

{'mean_ic': 0.01520684977220066, 'hac_se': 0.016147062727220118, 't_stat': 0.9417718893582744, 'p_value': 0.34642233143900913, 'n_periods': 2016, 'effective_lags': 20, 'naive_se': 0.005035672377891978, 'naive_t_stat': 3.019825086112238}


## Visualizing why the two t-stats disagree

The mean and the two t-stats are three numbers; this is the series behind
them. The daily points cluster into runs of several months each - visible
autocorrelation, not noise - which is exactly the dependence the naive
standard error ignores and the HAC correction accounts for.

In [7]:
from ml4t.diagnostic.results.signal_results.ic import SignalICResult
from ml4t.diagnostic.visualization.signal.ic_plots import plot_ic_ts
from scipy import stats as scipy_stats

ic_values = ic_series["ic"].to_numpy().astype(float)
naive_t, naive_p = scipy_stats.ttest_1samp(ic_values, 0)
assert abs(float(naive_t) - hac_stats["naive_t_stat"]) < 1e-6

ic_result = SignalICResult(
    ic_by_date={"21D": ic_values.tolist()},
    dates=[d.isoformat() for d in ic_series["timestamp"]],
    ic_mean={"21D": float(ic_values.mean())},
    ic_std={"21D": float(ic_values.std())},
    ic_t_stat={"21D": float(naive_t)},
    ic_p_value={"21D": float(naive_p)},
    ic_positive_pct={"21D": float((ic_values > 0).mean())},
    ic_ir={"21D": float(ic_values.mean() / ic_values.std())},
    ic_t_stat_hac={"21D": hac_stats["t_stat"]},
    ic_p_value_hac={"21D": hac_stats["p_value"]},
    hac_lags_used=hac_stats["effective_lags"],
)

ic_fig = plot_ic_ts(
    ic_result, period="21D", rolling_window=21, show_significance=True, theme="print"
)
export_static(ic_fig, "model-ic-series.png", width=1400, height=650, scale=2)
ic_fig

`hac_stats["mean_ic"]` is the honest headline number - not the naive
t-statistic you'd get from treating each daily IC observation as
independent. Because the label is a 21-day forward return, consecutive
daily ICs share ~20 days of the same underlying return window and are
mechanically autocorrelated; the naive t-stat overstates significance.
`label_horizon=21` tells the HAC correction the minimum lag to account
for, rather than relying on order selection alone.

**Run this notebook and the two t-stats disagree with each other**: naive
≈ 3.02 (nominally "significant"), HAC ≈ 0.94 (not significant). That
gap *is* the lesson, not a bug to fix - eight simple technical features
on a monthly-rebalanced ETF panel do not reliably beat noise once the
autocorrelation induced by the overlapping 21-day label is priced in.
A model with an IC this weak has no business going anywhere near a
backtest that claims a live edge; block 8 puts it through one anyway, on
purpose, to show what "weak signal, meet real costs" actually looks like.

## Feature importance

Cheap to compute, easy to over-read. Treat this as "what the last fold's
model leaned on," not as a causal or stable ranking across the full
18-year history - `13_model_analysis.py` in the full case study goes much
further (SHAP, permutation importance, stability across folds) than this
workshop has time for.

In [8]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance)

dollar_vol_rank    455
vol_63d            451
mom_252d           442
vol_21d            429
mom_126d           356
mom_63d            319
mom_21d            174
rsi_14             174
dtype: int32


**Next:** `04_backtest.ipynb` - turn these predictions into positions and
run a cost-aware backtest. Save the pooled out-of-sample predictions so
the backtest notebook doesn't need to retrain anything.

In [9]:
oos.to_parquet(f"{DATA_DIR}/oos_predictions.parquet", index=False)